In [89]:
import numpy as np
import jax
import jax.numpy as jnp
import optax
import unittest

In [98]:
def loss_fn(z, Sigma_tilde):
    """
        z: vector supporting K(z)
        Sigma_tilde: matrix of observed (privatized) covariance entries
    """
    # defines jax-numpy loss
    d = Sigma_tilde.shape[0]
    K_z = get_K(z, d)
    log_det_term = jnp.log(jnp.linalg.det(K_z))
    trace_term = jnp.trace(jnp.matmul(K_z, Sigma_tilde))
    #print(-log_det_term + trace_term)
    return -log_det_term + trace_term

def get_K(z, d):
    """
    z: vector supporting K(z)
    d: dimension of Sigma
    """
    z_mat = z.reshape((d, d))
    
    # get lower tri part w/ the diagonal
    lower_tri_indices = np.tril_indices(d)
    z_vals = z_mat[lower_tri_indices]
    q = len(z_vals)
    
    E1 = np.zeros((d, q))
    E2 = np.zeros((d, q))
    
    for k, (i_k, j_k) in enumerate(zip(*lower_tri_indices)):
        E1[i_k, k] = 1
        E2[j_k, k] = 1

    diag_z = jnp.diag(z_vals)
    
    K = E1 @ diag_z @ E2.T + E2 @ diag_z @ E1.T
    
    return K

def is_positive_definite(M):
    return jnp.all(jnp.linalg.eigvalsh(M) > 0)


def initialize_z(Sigma_tilde):
    """
        Sigma_tilde: matrix of observed (privatized) covariance entries
    """
    # Initialize z such that later K(z) can be defined as a diagonal matrix with positive entries
    z = np.zeros(Sigma_tilde.shape)
    diag_elements = np.maximum(0.01, 1.0 / np.diag(Sigma_tilde))
    diag_elements = diag_elements / 2  # avoid counting diagonal elements twice
    
    for i in range(len(diag_elements)):
        z[i, i] = diag_elements[i]

    print("Sigma_tilde", Sigma_tilde)
    print("z", z)

    z = z.ravel()
        
    return z

def binary_search_pd_point(z, grad):
    """
        z: current z vector
        grad: current gradient of the loss function wrt z
        step_size: initial step size for the search
        Sigma_tilde: matrix of observed (privatized) covariance entries
    """ 
    lower, upper = 0.0, 1.0
    while upper - lower > 1e-6:
        # search for mid s.t. K(z - mid*grad) is in the pos def cone
        mid = (lower + upper) / 2
        z_new = z - mid * grad
        if not is_positive_definite(get_K(z_new)):
            # not sure about this yet
            upper = mid
        else:
            lower = mid
    return z - upper * grad
    

def optimize_lbfgs(Sigma_tilde, z, lr=1e-1, max_iters=100):
    """
        Sigma_tilde: matrix of observed (privatized) covariance entries
        z: vector supporting K(z)
    """

    d = Sigma_tilde.shape[0]
    
    lr = 1e-1
    opt = optax.scale_by_lbfgs()
    
    # init state
    state = opt.init(z)

    # Run optimization
    
    for i in range(max_iters):
        loss, gradient = jax.value_and_grad(loss_fn)(z, Sigma_tilde)
        print(f'iter: {i}, loss: {loss:.2e}')
        updates, state = opt.update(gradient, state, z)
        z_new = z - lr * updates
        
        # enforce that z_new leads to a pos definite K(z_new), otherwise binary search
        if not is_positive_definite(get_K(z_new, d)):
            print("not PD, binary search for PD value")
            z_new = binary_search_pd_point(z, gradient)

        z = z_new

    print(get_K(z, d))
    
    return z


# helpers # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

def matrix_to_lower_triangular_dict(matrix):
    # cast z information as a mapping of lower triangular elements to their indices
    d = matrix.shape[0]  # Assuming square matrix
    lower_tri_dict = {}
    for row in range(d):
        for col in range(row + 1):
            if matrix[row, col]!=0:
                lower_tri_dict[(row, col)] = float(matrix[row, col])
    return lower_tri_dict

# tests # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

class TestAuxiliaryFunctions(unittest.TestCase):
    def test_K(self):
        d = 3
        z = np.array([[1/2., 0., 0.],
                      [0., 2/2., 0.],
                      [4., 0., 3/2.]]).flatten()
        K_result = get_K(z, d)
        K_true = np.array([[1., 0., 4.],
                           [0., 2., 0.],
                           [4., 0., 3.]])
        self.assertTrue(np.allclose(K_result, K_true, atol=1e-10))

    def test_initialize_z(self):
        Sigma_tilde = np.array([[1/1., 0., 1/4.],
                               [0., 1/2., 0.],
                               [1/4., 0., 1/3.]])
        d = Sigma_tilde.shape[0]
        
        z_mat = np.array([[1/2., 0., 0.],
                          [0., 2/2., 0.],
                          [4., 0., 3/2.]])  # in dict form, z_dict = {(0,0): 1/2, (1,1): 2/2, (2,2): 3/2, (2,0): 4}
        z_init = initialize_z(Sigma_tilde)
        z_init_mat = z_init.reshape((d, d))

        for i in range(d):
            self.assertAlmostEqual(z_init_mat[i,i], z_mat[i,i], places=10)

    def test_lower_triangular_dict(self):
        
        matrix = np.array([
            [1., 0., 4.],
            [0., 2., 0.],
            [4., 0., 3.]
        ])

        expected_dict = {
            (0, 0): 1.0,
            (1, 1): 2.0,
            (2, 0): 4.0,
            (2, 2): 3.0
        }

        result_dict = matrix_to_lower_triangular_dict(matrix)

        self.assertEqual(result_dict, expected_dict, "The resulting dictionary does not match the expected dictionary.")

In [99]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_K (__main__.TestAuxiliaryFunctions) ... ok
test_initialize_z (__main__.TestAuxiliaryFunctions) ... ok
test_lower_triangular_dict (__main__.TestAuxiliaryFunctions) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.007s

OK


Sigma_tilde [[1.         0.         0.25      ]
 [0.         0.5        0.        ]
 [0.25       0.         0.33333333]]
z [[0.5 0.  0. ]
 [0.  1.  0. ]
 [0.  0.  1.5]]


In [100]:
Sigma_tilde = np.array([[1/1., 0., 1/4.],
                               [0., 1/2., 0.],
                               [1/4., 0., 1/3.]])
z_init = initialize_z(Sigma_tilde)
optimize_lbfgs(Sigma_tilde, z_init, lr=1e-1, max_iters=30)

Sigma_tilde [[1.         0.         0.25      ]
 [0.         0.5        0.        ]
 [0.25       0.         0.33333333]]
z [[0.5 0.  0. ]
 [0.  1.  0. ]
 [0.  0.  1.5]]
iter: 0, loss: 1.21e+00


/Users/ceciliaferrando/Documents/UMASS/RESEARCH/GaussianPGM/privateggm/lib/python3.10/site-packages/optax/_src/transform.py:1412: UserWarning: Explicitly requested dtype float64 requested in zeros is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/google/jax#current-gotchas for more.
  lambda leaf: jnp.zeros((memory_size,) + leaf.shape, dtype=leaf.dtype),


iter: 1, loss: 1.18e+00
iter: 2, loss: 1.15e+00
iter: 3, loss: 1.13e+00
iter: 4, loss: 1.10e+00
iter: 5, loss: 1.08e+00
iter: 6, loss: 1.07e+00
iter: 7, loss: 1.06e+00
iter: 8, loss: 1.05e+00
iter: 9, loss: 1.04e+00
iter: 10, loss: 1.03e+00
iter: 11, loss: 1.03e+00
iter: 12, loss: 1.02e+00
iter: 13, loss: 1.02e+00
iter: 14, loss: 1.02e+00
iter: 15, loss: 1.01e+00
iter: 16, loss: 1.01e+00
iter: 17, loss: 1.01e+00
iter: 18, loss: 1.01e+00
iter: 19, loss: 1.01e+00
iter: 20, loss: 1.01e+00
iter: 21, loss: 1.00e+00
iter: 22, loss: 1.00e+00
iter: 23, loss: 1.00e+00
iter: 24, loss: 1.00e+00
iter: 25, loss: 1.00e+00
iter: 26, loss: 1.00e+00
iter: 27, loss: 1.00e+00
iter: 28, loss: 1.00e+00
iter: 29, loss: 1.00e+00
[[ 1.2061479   0.         -0.86915165]
 [ 0.          2.0000126   0.        ]
 [-0.86915165  0.          3.6171436 ]]


Array([ 0.60307395,  0.        ,  0.        ,  0.        ,  1.0000063 ,
        0.        , -0.86915165,  0.        ,  1.8085718 ], dtype=float32)